fact orders

In [0]:
from pyspark.sql.functions import *

df = spark.sql("select * from databrick_cata.silver.orders_silver")
df.limit(10).display()

order_id,customer_id,product_id,order_date,quantity,total_amount,_rescued_data
O00001,C00710,P0159,2023-03-22,3,2022.87,null
O00002,C00954,P0036,2023-06-30,2,3560.74,null
O00003,C01578,P0427,2023-11-06,3,5903.52,null
O00004,C00962,P0332,2024-02-27,3,4107.99,null
O00005,C00156,P0038,2024-10-13,5,5784.95,null
O00006,C00521,P0174,2023-05-17,5,407.75,null
O00007,C00982,P0352,2024-01-18,4,4907.64,null
O00008,C00976,P0172,2023-01-10,4,7037.88,null
O00009,C01001,P0238,2023-04-20,3,4076.97,null
O00010,C00702,P0258,2023-07-07,4,5695.64,null


In [0]:
fd_dim_customer = spark.sql("select DimCustomerKey , customer_id as dim_customer_id from databrick_cata.gold.dimcustomers")
fd_dim_product = spark.sql("select product_id as DimProductKey, product_id as dim_product_id from databrick_cata.gold.dimproducts")

In [0]:
df_fact = df.join(fd_dim_customer, df['customer_id'] == fd_dim_customer['dim_customer_id'],how='left').join(fd_dim_product, df['product_id'] == fd_dim_product['dim_product_id'],how='left')


In [0]:
df_fact_new = df_fact.drop('customer_id','product_id','dim_product_id','product_id',"customer_id","product_id","dim_customer_id","dim_product_id","dim_customer_id")
df_fact_new.display()

order_id,order_date,quantity,total_amount,_rescued_data,DimCustomerKey,DimProductKey
O00001,2023-03-22,3,2022.87,null,710,P0159
O00002,2023-06-30,2,3560.74,null,954,P0036
O00003,2023-11-06,3,5903.52,null,1578,P0427
O00004,2024-02-27,3,4107.99,null,962,P0332
O00005,2024-10-13,5,5784.95,null,156,P0038
O00006,2023-05-17,5,407.75,null,521,P0174
O00007,2024-01-18,4,4907.64,null,982,P0352
O00008,2023-01-10,4,7037.88,null,976,P0172
O00009,2023-04-20,3,4076.97,null,1001,P0238
O00010,2023-07-07,4,5695.64,null,702,P0258


In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("databrick_cata.gold.FactOrders"):
    
    dlt_obj = DeltaTable.forName(spark, "databrick_cata.gold.FactOrders")

    dlt_obj.alias("trg").merge(df_fact_new.alias("src"), "trg.order_id = src.order_id AND trg.DimCustomerKey = src.DimCustomerKey AND trg.DimProductKey = src.DimProductKey")\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()

else:
    df_fact_new.write.format("delta")\
            .option("path","abfss://gold@datalakezakariae2026.dfs.core.windows.net/FactOrders")\
            .saveAsTable("databrick_cata.gold.FactOrders")

In [0]:
%sql
select * from databrick_cata.gold.FactOrders

order_id,order_date,quantity,total_amount,_rescued_data,DimCustomerKey,DimProductKey
O00001,2023-03-22,3,2022.87,null,710,P0159
O00002,2023-06-30,2,3560.74,null,954,P0036
O00003,2023-11-06,3,5903.52,null,1578,P0427
O00004,2024-02-27,3,4107.99,null,962,P0332
O00005,2024-10-13,5,5784.95,null,156,P0038
O00006,2023-05-17,5,407.75,null,521,P0174
O00007,2024-01-18,4,4907.64,null,982,P0352
O00008,2023-01-10,4,7037.88,null,976,P0172
O00009,2023-04-20,3,4076.97,null,1001,P0238
O00010,2023-07-07,4,5695.64,null,702,P0258
